In [1]:
import shapely
import os
from os import path
from math import sqrt, dist

import pandas as pd

import folium

## County Coverage
### Correlate my ride to LOJIC centerlines data
#### Data Sources
* My rides
    * You know where most lines start and end.
* LOJIC Data: Centerlines
    * find centerlines that only have two points. Try to match those first
    * then match longer lines.



In [2]:
"/Users/bencampbell/code/county_coverage/data/cleaner/centerline_coordinates.csv" 
"/Users/bencampbell/code/county_coverage/data/cleaner/centerline_data.csv"

CENTERLINE__DIR = "../data/cleaner"
CENTERLINE_LL = 'centerline_coordinates.csv' # longitudes, latitudes
CENTERLINE_DATA = 'centerline_data.csv' # tabular info

def get(VAR):
    P = path.join(CENTERLINE__DIR, VAR)
    assert(path.exists(P))
    #print(P)
    return pd.read_csv(P, index_col='OBJECTID')


da = get(CENTERLINE_DATA)


AssertionError: 

In [ ]:

def process_coordinate_string(costring):
    pairs = list()
    for LL in costring.split():
        long, lat = LL.split(',')
        pairs.append((float(long), float(lat)))
    return shapely.LineString(pairs)

def load_coordinates():
    co = get(CENTERLINE_LL)
    co['COORDINATES'] = co.COORDINATES.apply(process_coordinate_string)
    return co

CO = load_coordinates()

In [ ]:
bo=CO.COORDINATES[1].boundary
bo = (bo.bounds)
bo

def boundary_contains_point(boundary, point):
    longitude, latitude = point
    if boundary[0] <= longitude <= boundary[2]:
        if boundary[1] <= latitude <= boundary[3]:
            return True
    return False

def boundary_covers_other(boundary, other_boundary):
    xmin, ymin, xmax, ymax = boundary
    if xmin <= other_boundary[0] <= xmax:
        if xmin <= other_boundary[2] <= xmax:
            if ymin <= other_boundary[1] <= ymax:
                if ymin <= other_boundary[2] <= ymax:
                    return True
    return False

def point_distance(pointA, pointB):
    xdiff = pointB[0] - pointA[0]
    ydiff = pointB[1] - pointA[1]
    sign = -1 if xdiff < 0 else 1
    sign += -1 if ydiff < 0 else 1
    return sign * sqrt(xdiff + ydiff)

def find_closest_point(point_query, object):
    """Find closest point in object to point_query."""
    qx, qy = point_query
    minimum = 0
    memo = list()
    for index, point in enumerate(object):
        delta = ((point[0] - qx) ** 2) + ((point[1] - qy) ** 2)
        if delta == minimum:
            memo.append(index)
        elif delta < minimum:
            minimum = delta
            del memo[:]
            memo.append(index)
    return memo
        
    
    
    


In [ ]:

def delta(x, y):
    return ((x - qx) ** 2) + ((y - qy) ** 2)


#[1,2,3,4,3,2,4,5].index(3, 5)

In [ ]:
len(CO)- 2**16
from random import getrandbits
n=getrandbits(16)
print(f"{n} -> {n:016b}")

49917 -> 1100001011111101


In [ ]:
t = CO.COORDINATES.loc[172238]
CO.COORDINATES.apply(lambda x:hash(x.bounds)).is_unique # True! This is helpful,
# none of the roads have overlapping regions, which makes sense. 

boundaries = pd.DataFrame.from_records(CO.COORDINATES.apply(lambda x:x.bounds).values, columns='minx miny maxx maxy'.split(), index=CO.index)
boundaries



,minx,miny,maxx,maxy
OBJECTID,,,,
1,-85.681235,38.158161,-85.680950,38.158867
2,-85.801369,38.229343,-85.801201,38.230564
3,-85.805015,38.227538,-85.804692,38.228933
4,-85.680205,38.247608,-85.679863,38.248067
5,-85.742040,38.211815,-85.741648,38.211969
...,...,...,...,...
172237,-85.569499,38.334101,-85.568968,38.334777
172238,-85.568968,38.333488,-85.568469,38.334101
172239,-85.813102,38.261615,-85.812529,38.262190


In [ ]:
sifs = [c for c in da.columns if 'SIF' in c]
no_aliases = [a for a in sifs if 'ALIAS' not in a] 

# search pattern

get first road in ride path # <- wite code to make this work.
follow ride points along roads to trace the whole ride
    use cues from the sifids to build road network for searching. # <- next step is to build search network.

In [ ]:


def avg_point(points):
    size = len(points)
    return tuple(map(lambda xs:sum(xs)/size, zip(*points)))

def point_avg(points):
    n = len(points)
    xs, ys = zip(*points)
    return sum(xs)/n, sum(ys)/n

t = CO.COORDINATES[800]
point_avg(t.coords), avg_point(t.coords)



((-85.57733446262796, 38.165020496609664),
 (-85.57733446262796, 38.165020496609664))

In [ ]:

def make_centerline_map(line, tooltip, lineweight=5, linecolor='#ff0000'):
    center = line.centroid
    line = [(y,x) for (x, y) in line.coords]
    carte = folium.Map(location=(center.y, center.x), zoom_start=15, scrollWheelZoom=False)
    folium.PolyLine(locations=line, color=linecolor, weight=lineweight,
                    tooltip=tooltip).add_to(carte)
    return carte
    

def make_figure(carte, title:str, height=200, weight=200):
    figure = folium.Figure(height=height, width=weight)
    title_html = f'''
    <h3 style="margin-left:10px; 
    font-size:14px; font-weight: bold; 
    text-align: center; color: #555555;">{title}</h3>'''
    figure.get_root().html.add_child(folium.Element(title_html))
    carte.add_to(figure)
    return figure

def show_centerline(index):
    try:
        line_record = CO.COORDINATES[index]
    except KeyError:
        raise IndexError(f"No such centerline record with index {index}")
    roadname = da.loc[index]['ROADNAME']

    return make_figure(make_centerline_map(line_record, roadname), F"{roadname}\t({index})")
# messy functioncalls/paths clean it up. 

show_centerline(100)

In [ ]:
import random
indexes = list(CO.COORDINATES.index)
def random_centerline():
    return show_centerline(random.choice(indexes))
random_centerline()



In [ ]:
def add_line_to_map(centerline_coords, name, carte, color='##00FF00', lineweight=5):
    folium.PolyLine(locations=[(y,x) for x, y in centerline_coords], color=color, weight=lineweight,
                    tooltip=name).add_to(carte)
    return carte

def make_map_with_crossroads(index):
    line = CO.COORDINATES[index]
    record = da.loc[index]
    name = record['ROADNAME']
    carte = make_centerline_map(line, name)


r=make_map_with_crossroads(100)
da.loc[100][[col for col in da.columns if 'SIF' in col]]

SIFID           5218
SIFCODE         5743
SIFIDLOW        8594
LOCROSSSIF      9811
SIFIDHI         6236
HICROSSSIF      6931
ALIAS1_SIFID       0
ALIAS2_SIFID       0
ALIAS3_SIFID       0
ALIAS4_SIFID       0
Name: 100, dtype: object

In [ ]:
# Match each code to a roadname/objectid
def get_sif_info(index):
    return da.loc[index][[col for col in da.columns if 'SIF' in col]]

def get_sfs(code) -> "OBJECTID":
    out = dict()
    columns = ['ROADNAME', 'SIFID', 'SIFCODE']
    sifid = da[da.SIFID == code][columns]
    sifcode = da[da.SIFCODE == code][columns]
    #if len(sifid):
    out['SIFID'] = sifid
    #if len(sifcode):
    out['SIFCODE'] = sifcode
    return out


#display(get_sif_info(100))

d=get_sfs(6931)
d

{'SIFID':                  ROADNAME  SIFID SIFCODE
 OBJECTID                                
 18175     WOLFPEN GLEN CT   6931    7741,
 'SIFCODE': Empty DataFrame
 Columns: [ROADNAME, SIFID, SIFCODE]
 Index: []}

In [ ]:
def get_sif(sifid=None, sifcode=None):
    if sifid is not None:
        return da[da.SIFID == sifid].index
    if sifcode is not None:
        return da[da.SIFCODE == sifcode].index

get_sif(6931)

Index([18175], dtype='int64', name='OBJECTID')

In [ ]:

da[da.SIFID == 6881]['ROADNAME']


OBJECTID
27648    CLIFFWOOD HILL WAY
32683    CLIFFWOOD HILL WAY
Name: ROADNAME, dtype: object

In [ ]:
def getinfo(index):
       return da.loc[index][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

r = getinfo(32683)
r

ROADNAME      CLIFFWOOD HILL WAY
SIFID                       6881
SIFCODE                     7690
SIFIDLOW                    1196
LOCROSSSIF                  1182
LOCROSSPR                    NaN
LOCROSSNA              CLIFFWOOD
LOCROSSSU                    AVE
SIFIDHI                     8594
HICROSSSIF                  9811
HICROSSPR                    NaN
HICROSSNA               DEAD END
HICROSSSU                    NaN
Name: 32683, dtype: object

In [ ]:
da[da.SIFCODE == '7690'][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

,ROADNAME,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,LOCROSSPR,LOCROSSNA,LOCROSSSU,SIFIDHI,HICROSSSIF,HICROSSPR,HICROSSNA,HICROSSSU
OBJECTID,,,,,,,,,,,,,
27648,CLIFFWOOD HILL WAY,6881,7690,8594,9811,NaN,DEAD END,NaN,1196,1182,NaN,CLIFFWOOD,AVE
32683,CLIFFWOOD HILL WAY,6881,7690,1196,1182,NaN,CLIFFWOOD,AVE,8594,9811,NaN,DEAD END,NaN


In [ ]:
M = make_centerline_map(CO.COORDINATES[27648], da.loc[27648].ROADNAME)

#folium.PolyLine(locations=[(y,x) for x, y in CO.COORDINATES[32683].coords], color="#AA0000", weight=5,
 #                   tooltip=da.loc[32683].ROADNAME).add_to(M)

def add_line_to_map(index, carte, color):
    folium.PolyLine(locations=[(y,x) for x, y in CO.COORDINATES[index].coords], color=color, weight=5,
                    tooltip=da.loc[index].ROADNAME).add_to(carte)
    
add_line_to_map(32683, M, color='#AA0000')
add_line_to_map(16733, M, color='#00aaaa')


M

In [ ]:
da[da.SIFID == 1196][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

,ROADNAME,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,LOCROSSPR,LOCROSSNA,LOCROSSSU,SIFIDHI,HICROSSSIF,HICROSSPR,HICROSSNA,HICROSSSU
OBJECTID,,,,,,,,,,,,,
27699,CLIFFWOOD AVE,1196,1182,441,0423,S,BAYLY,AVE,6881,7690,NaN,CLIFFWOOD HILL,WAY


In [ ]:
add_line_to_map(27699, M, '#00aaaa')

In [ ]:
F=make_figure(M, "SIFCODE=7690")
F

In [ ]:
da[da.SIFCODE == '1182'][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

,ROADNAME,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,LOCROSSPR,LOCROSSNA,LOCROSSSU,SIFIDHI,HICROSSSIF,HICROSSPR,HICROSSNA,HICROSSSU
OBJECTID,,,,,,,,,,,,,
27699,CLIFFWOOD AVE,1196,1182,441,0423,S,BAYLY,AVE,6881,7690,NaN,CLIFFWOOD HILL,WAY


In [ ]:
add_line_to_map(21414, M, '#00FF00')
add_line_to_map(27699, M, '#00FF00')

F

In [ ]:
F

In [ ]:
da[da.SIFCODE == '8594'][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

,ROADNAME,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,LOCROSSPR,LOCROSSNA,LOCROSSSU,SIFIDHI,HICROSSSIF,HICROSSPR,HICROSSNA,HICROSSSU
OBJECTID,,,,,,,,,,,,,
11345,GREEN GARDEN CT,7548,8594,8594,9811,NaN,DEAD END,NaN,2443,2471,NaN,VELDEN,DR
24894,GREEN GARDEN CT,7548,8594,2443,2471,NaN,VELDEN,DR,12261,6005,NaN,SIX MILE,LN


In [ ]:
da[da.SIFCODE == '9811'][['ROADNAME', 'SIFID', 'SIFCODE','SIFIDLOW', 'LOCROSSSIF', 'LOCROSSPR',
       'LOCROSSNA', 'LOCROSSSU', 'SIFIDHI', 'HICROSSSIF', 'HICROSSPR',
       'HICROSSNA', 'HICROSSSU']]

,ROADNAME,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,LOCROSSPR,LOCROSSNA,LOCROSSSU,SIFIDHI,HICROSSSIF,HICROSSPR,HICROSSNA,HICROSSSU
OBJECTID,,,,,,,,,,,,,


In [ ]:
codes = set(da.SIFCODE.unique())
ids = set(int(d) for d in da.SIFID.unique())
da[['SIFCODE', 'SIFID']]

,SIFCODE,SIFID
OBJECTID,,
1,9887,8665
2,6570,5926
3,0458,473
4,2470,2442
5,4974,4573
...,...,...
172237,F416,15082
172238,F416,15082
172239,F714,15566


In [ ]:
codes
strings = set()

def conv(x):
    try:
        return int(x)
    except:
        strings.add(x)
        return x
    
conv_codes = {conv(c) for c in codes}

def conv_test(x):
    return int(x, 16)

da.SIFCODE.apply(lambda x:int(x, 16))
conv_test('F416')

62486

In [ ]:
def do(ls):
    xmin, ymin, xmax, ymax = ls.bounds
    return (xmin, xmax), (ymin, ymax)
boundaries = pd.DataFrame.from_records(CO.COORDINATES.apply(lambda ls:ls.bounds).to_list(),
                                        columns='xmin ymin xmax ymax'.split(), 
                                        index=CO.index)

boundaries

,xmin,ymin,xmax,ymax
OBJECTID,,,,
1,-85.681235,38.158161,-85.680950,38.158867
2,-85.801369,38.229343,-85.801201,38.230564
3,-85.805015,38.227538,-85.804692,38.228933
4,-85.680205,38.247608,-85.679863,38.248067
5,-85.742040,38.211815,-85.741648,38.211969
...,...,...,...,...
172237,-85.569499,38.334101,-85.568968,38.334777
172238,-85.568968,38.333488,-85.568469,38.334101
172239,-85.813102,38.261615,-85.812529,38.262190


In [ ]:
xmin = -85.9424052123031
xmax = -85.3474509339146

avg = (xmin + xmax)/2

xs = boundaries['xmin xmax'.split()]
ys = boundaries['ymin ymax'.split()]
xs, ys


(               xmin       xmax
 OBJECTID                      
 1        -85.681235 -85.680950
 2        -85.801369 -85.801201
 3        -85.805015 -85.804692
 4        -85.680205 -85.679863
 5        -85.742040 -85.741648
 ...             ...        ...
 172237   -85.569499 -85.568968
 172238   -85.568968 -85.568469
 172239   -85.813102 -85.812529
 172240   -85.761629 -85.761182
 172545   -85.763058 -85.762272
 
 [34874 rows x 2 columns],
                ymin       ymax
 OBJECTID                      
 1         38.158161  38.158867
 2         38.229343  38.230564
 3         38.227538  38.228933
 4         38.247608  38.248067
 5         38.211815  38.211969
 ...             ...        ...
 172237    38.334101  38.334777
 172238    38.333488  38.334101
 172239    38.261615  38.262190
 172240    38.218800  38.220560
 172545    38.219393  38.219482
 
 [34874 rows x 2 columns])

In [ ]:
ymidpoint = ys.sort_values(by='ymin').iloc[len(ys)//2]
ys.mean(), ymidpoint

(ymin    38.205831
 ymax    38.206809
 dtype: float64,
 ymin    38.210839
 ymax    38.212324
 Name: 10974, dtype: float64)

In [ ]:
class bound(float):
    def to_right(self, other):
        """Return True if other is to the right of boundary"""
        return self < other[0]

    def is_in(self, other):
        return other[0] <= self <= other[1]
        
    def to_left(self, other):
        return self > other[1]
    
    def test(self, other):
        return (self.to_left(other), self.is_in(other), self.to_right(other))
    
def split_records(rs, cut):
    """rs[rs.apply(cut) == True vs False]"""


In [ ]:
for a, b in ys.iterrows():
    ...
    
def estimate(mins, maxs):
    return (min(mins) + max(maxs)) / 2






,ymin,ymax
OBJECTID,,
1,38.158161,38.158867
2,38.229343,38.230564
3,38.227538,38.228933
4,38.247608,38.248067
5,38.211815,38.211969
...,...,...
172237,38.334101,38.334777
172238,38.333488,38.334101
172239,38.261615,38.262190


In [ ]:

a=ys.ymin.combine(ys.ymax, lambda x, y:(x+y)/2).mean()
a=float(a)    
a += 0.0045

def estimate(points):
    return points.median().mean()

def tester(e):
    return lambda u, w:(w <= e) or (e >= u)

def test_series(series, cols):
    mincol, maxcol = cols.split()
    #sin = series[mincol].combine(series[maxcol],


yin = ys.ymin.combine(ys.ymax, tester(a))
yin
no = ys[yin == False]
yes = ys[yin == True]
len(yes), len(no), len(yes) - len(no), a

# found yaxis median 1

class find_good_cut:
    def __init__(self, spreads):
        e = sum(map(sum, zip(*spreads)))
        e /= len(spreads)
        e /= 2
        self.estimate = e
        self.offset = 0.01

    


In [ ]:
e = float(xs.median().mean())
e -= 0.0007

def test(u, w):
    return (w <= e) or (e >= u)

xin = xs.xmin.combine(xs.xmax, test)
xin
no = xs[xin == False]
yes = xs[xin == True]
len(yes), len(no), len(yes)-len(no), e

# found x axis median 1

(17436, 17438, -2, -85.66701846358393)

In [ ]:
from itertools import pairwise
from math import floor

class PointEncoderDecoder:
    def __init__(self, xmin, xmax, ymin, ymax):
        self.xmin, self.xmax = xmin, xmax
        self.ymin, self.ymax = ymin, ymax

    def value_encoder(self, value, vlow, vhigh, precision=None):
        if precision is None:
            while 1:
                cut = (vlow + vhigh) / 2
                if value < cut:
                    yield 0
                    vhigh = cut
                elif value == cut:
                    yield 0
                    break
                else:
                    yield 1
                    vlow = cut
        else:
            while precision:
                cut = (vlow + vhigh) / 2
                if value <= cut:
                    yield 0
                    vhigh = cut
                elif value == cut:
                    yield 0
                    break
                else:
                    yield 1
                    vlow = cut
                precision -= 1
        return


        



    def encode(self, point, precision=None):
        x, y = point
        xlow, xhigh = self.xmin, self.xmax


        ylow, yhigh = self.ymin, self.ymax



    def decode_box(self, bits:int):
        xlow, xhigh = self.xmin, self.xmax
        ylow, yhigh = self.ymin, self.ymax
        mask = 1 << (bits.bit_length() - 1)
        axis = True 
        while mask:
            # True :: east, north, right, up
            # False :: west, south, left, down
            if axis: # x axis
                xcut = (xlow + xhigh)/2
                if bits & mask: # east of cut
                    xlow = xcut
                else: # west of cut
                    xhigh = xcut
                axis = False # switch axis
            else: # y axis
                ycut = (ylow + yhigh)/2
                if bits & mask: # north of cut
                    ylow = ycut
                else: # south of cut
                    yhigh = ycut
                axis = True # switch axis
            mask >>= 1
        return {"xlow":xlow, "xhigh":xhigh, "ylow":ylow, "yhigh":yhigh}
    
    def box_center(self, xlow, xhigh, ylow, yhigh):
        return ((xlow + xhigh) / 2, (ylow + yhigh) / 2)

        

def cut_gen(vlow, vhigh):
    return (vlow + vhigh)/2



        

In [ ]:
class Encoding(int):
    

SyntaxError: expected default value expression (1131401597.py, line 10)

In [ ]:
def cut(a, b):
    return (a + b)/2

class PointEncoderDecoder:
    _StandardPrecision = 8

    def __init__(self, xMin:float, xMax:float, yMin:float, yMax:float):
        self.xMin, self.xMax = xMin, xMax
        self.yMin, self.yMax = yMin, yMax

    def encode(self, point:(float, float), precision=_StandardPrecision) -> int:
        out = 0
        x, y = point
        xLow, xHigh = self.xMin, self.xMax
        yLow, yHigh = self.yMin, self.yMax
        while precision:
            out <<= 2
            xCut = (xLow + xHigh)/2
            if x > xCut:
                out += 0b10
            yCut = (yLow + yHigh)/2
            if y > yCut:
                out += 0b01
            precision -= 1
        return out
            
    def decode(self, encoding:int) -> (float, float):
        xLow, xHigh = self.xMin, self.xMax
        yLow, yHigh = self.yMin, self.yMax
        mask = 1 << (encoding.bit_length() - 1)
        while mask:
            # x dimension
            xCut = (xLow + xHigh) / 2
            if mask & encoding:
                xLow = xCut
            else:
                xHigh = xCut
            # y dimension
            mask >>= 1
            yCut = (yLow + yHigh) / 2
            if mask & encoding:
                yLow = yCut
            else:
                yHigh = yCut
        return (xCut, yCut)
    
    def get_terminal_frame(self, encoding:int):
        xLow, xHigh = self.xMin, self.xMax
        yLow, yHigh = self.yMin, self.yMax
        mask = 1 << (encoding.bit_length() - 1)
        while mask:
            # x dimension
            xCut = (xLow + xHigh) / 2
            if mask & encoding:
                xLow = xCut
            else:
                xHigh = xCut
            # y dimension
            mask >>= 1
            yCut = (yLow + yHigh) / 2
            if mask & encoding:
                yLow = yCut
            else:
                yHigh = yCut
        return {"xLow":xLow, "xHigh":xHigh,
                "yLow":yLow, "yHigh":yHigh}        
                



In [ ]:
x=237831232
from math import ceil
x.to_bytes(ceil(x.bit_length()/8))

b'\x0e-\x04@'

In [ ]:
LONGITUDES = LONG_MIN, LONG_MAX = -85.94712712079293, -85.40492183942492
LATITUDES = LAT_MIN, LAT_MAX = 37.99712528351634, 38.38023822809115

DELTA_LONG = LONG_MAX - LONG_MIN  # == 0.5422052813680125
DELTA_LAT = LAT_MAX - LAT_MIN  # == 0.3831129445748118

def translate_point(longitude, latitude):
    return ((longitude - LONG_MIN) / DELTA_LONG), ((latitude - LAT_MIN) / DELTA_LAT)

def to_surreal(value:float, precision:int=None):
    ...



In [ ]:
class bits(int):
    def __repr__(self):
        return bin(self)

def to_surreal_AP(value:float):
    assert 0 <= value <= 1
    cut = 0.5
    approx = 0.5
    out = 1
    while True:
        cut /= 2
        out <<= 1
        if value > approx:
            out += 1
            approx += cut
        elif value != approx:
            #out += 0
            approx -= cut
        else:
            break
    return bits(out)

def to_surreal_bits(value:float):
    assert 0 <= value <= 1
    cut = 0.5
    approx = 0.5
    while True:
        if value > approx:
            yield cut
            cut /= 2
            approx += cut
        elif value != approx:
            cut /= 2
            approx -= cut
        else:
            yield cut
            break

list(to_surreal_bits(0.751))




[0.0078125,
 0.001953125,
 0.0001220703125,
 6.103515625e-05,
 3.0517578125e-05,
 1.52587890625e-05,
 3.814697265625e-06,
 9.5367431640625e-07,
 4.76837158203125e-07,
 2.384185791015625e-07,
 7.450580596923828e-09,
 1.862645149230957e-09,
 1.1641532182693481e-10,
 5.820766091346741e-11,
 2.9103830456733704e-11,
 1.4551915228366852e-11,
 3.637978807091713e-12,
 9.094947017729282e-13,
 4.547473508864641e-13,
 2.2737367544323206e-13,
 7.105427357601002e-15,
 1.7763568394002505e-15,
 1.1102230246251565e-16,
 5.551115123125783e-17,
 2.7755575615628914e-17,
 1.3877787807814457e-17,
 3.469446951953614e-18,
 1.734723475976807e-18]

In [ ]:
class gridcode(int):
    @classmethod
    def axis_from_float(cls, value:float, precision=8) -> int:
        assert 0 <= value <= 1
        cut = 0.25
        approx = 0.5
        while precision:
            if value > approx:
                yield 1
                approx += cut
            else: # value <= approx
                yield 0
                approx -= cut
            cut /= 2
            precision -= 1
        return new
    
    @classmethod
    def from_coordinates(cls, longitude, latitude,, precision=8):
        out = 0
        for x, y in zip(self.axis_from_float(longitude), self.axis_from_float(latitude)):
            out <<= 2
            if x:
                out += 2
            if y:
                out += 1
                

    
    def axis_estimate(self):
        size = self.bit_length() - 1
        mask = 1 << (size - 1)
        approx = 0.5
        place = bool(mask & self)
        mask >>= 1
        cut = 0.25
        while mask:
            if place:
                approx += cut
            else:
                approx -= cut
            cut /= 2
            place = bool(mask & self)
            mask >>= 1
        if mask & 1:
            return approx - cut , approx
        else:
            return approx, approx + cut
      
    
    def __repr__(self):
        return bin(self)


gridcode.from_float(0.251, precision=19).grid_estimate()


IndentationError: expected an indented block after function definition on line 19 (3945763372.py, line 22)

In [ ]:
def encode_axis(x:float, precision=8):
    assert 0 <= x <= 1
    approx = 0.5
    cut = 0.5
    while precision:
        cut /= 2
        if x > approx:
            yield 1
            approx += cut
        else:
            yield 0
            approx -= cut
        precision -= 1


class gridcode(int):
    def __new__(self, longitude, latitude, *, precision=8):
        new = 1
        for x, y in zip(encode_axis(longitude, precision=precision), encode_axis(latitude, precision=precision)):
            new << 2
            if x:
                new += 0b10
            if y:
                new += 0b01
        return super().__new__(gridcode, new)
    
    def get_box(self):
        this = self
        while this:
            y = this & 1
            x = (this & 2) >> 1
    
    def __repr__(self):
        return bin(self)

In [ ]:
list(encode_axis(0.75))

[1, 0, 1, 1, 1, 1, 1, 1]

In [ ]:
    
def decode_surreal(surr:int):
    size = surr.bit_length() - 2
    out = 0
    while size:
        if 1 & surr:
            out += (1/(2**size))
        size -= 1
        surr >>= 1
    return out
bool(None)

False